# Cost Analysis

Reads `reactor_usage_events` (the source of truth for billing) and `reactor_user_budgets`. Produces:

- Cost per conversation
- Cost per user across day/week/month windows
- Provider/model cost breakdown
- Top spenders
- Budget vs actual

The notebook is parameterized via [papermill](https://papermill.readthedocs.io). The cell below tagged `parameters` is overridden by `papermill -p start_date 2026-04-01 -p end_date 2026-04-29 ...`.

It does **not** recompute pricing from `providers.yaml` — it sums `costs.totalUsdCents` exactly as the runtime `ReactoryUsageService` wrote them. This keeps the notebook immune to YAML drift.

In [ ]:
# ── Parameters (override via papermill) ──────────────────────────────────────
CONFIG_KEY = "reactory"
ENVIRONMENT_KEY = "local"
START_DATE = None     # ISO date string e.g. "2026-04-01"; None = beginning of time
END_DATE = None       # ISO date string; None = now
USER_ID = None        # restrict to single user (hex ObjectId) or None for all
OUTPUT_DIR = None     # directory to dump CSV/PNG artifacts into; None = no output

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
import os, warnings
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from bson import ObjectId
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [ ]:
# ── Load environment & connect to Mongo ──────────────────────────────────────
SERVER_ROOT = Path(__file__).resolve().parents[4] if "__file__" in dir() else Path.cwd().parents[4]
env_file = SERVER_ROOT / "config" / CONFIG_KEY / f".env.{ENVIRONMENT_KEY}"
if env_file.exists():
    load_dotenv(env_file)
MONGO_URI = os.environ["MONGO_URI"]

client = MongoClient(MONGO_URI)
db = client.get_default_database()
events = db["reactor_usage_events"]
budgets = db["reactor_user_budgets"]

print("Connected to:", db.name)
print("Events:", events.estimated_document_count(), "| Budgets:", budgets.estimated_document_count())


In [ ]:
# ── Build query window ───────────────────────────────────────────────────────
match = {}
if START_DATE or END_DATE:
    occurred = {}
    if START_DATE:
        occurred["$gte"] = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc)
    if END_DATE:
        occurred["$lt"] = datetime.fromisoformat(END_DATE).replace(tzinfo=timezone.utc)
    match["occurredAt"] = occurred
if USER_ID:
    match["userId"] = ObjectId(USER_ID)

print("Query match:", match)


In [ ]:
# ── Load events into a dataframe ─────────────────────────────────────────────
rows = list(events.find(match, {
    "_id": 0,
    "userId": 1, "conversationId": 1, "providerId": 1, "modelId": 1,
    "usage": 1, "costs": 1, "pricingSource": 1, "occurredAt": 1,
}))
if not rows:
    print("No events match the filter.")
    df = pd.DataFrame()
else:
    df = pd.json_normalize(rows)
    df["occurredAt"] = pd.to_datetime(df["occurredAt"], utc=True)
    df["day"] = df["occurredAt"].dt.tz_convert("UTC").dt.date
    df["week"] = df["occurredAt"].dt.tz_convert("UTC").dt.to_period("W-MON").dt.start_time
    df["month"] = df["occurredAt"].dt.tz_convert("UTC").dt.to_period("M").dt.start_time
df.head()


## Cost per conversation

In [ ]:
if not df.empty:
    by_conv = df.groupby(["conversationId", "providerId", "modelId"]).agg(
        total_usd_cents=("costs.totalUsdCents", "sum"),
        prompt_tokens=("usage.promptTokens", "sum"),
        completion_tokens=("usage.completionTokens", "sum"),
        events=("costs.totalUsdCents", "size"),
    ).reset_index().sort_values("total_usd_cents", ascending=False)
    display(by_conv.head(20))


## Cost per user (day / week / month)

In [ ]:
if not df.empty:
    for period in ["day", "week", "month"]:
        agg = df.groupby(["userId", period])["costs.totalUsdCents"].sum().reset_index()
        agg = agg.rename(columns={"costs.totalUsdCents": "usd_cents"})
        print(f"\n=== Cost per user per {period} (top 20 rows) ===")
        display(agg.sort_values("usd_cents", ascending=False).head(20))


## Provider / model breakdown

In [ ]:
if not df.empty:
    by_model = df.groupby(["providerId", "modelId"]).agg(
        total_usd_cents=("costs.totalUsdCents", "sum"),
        events=("costs.totalUsdCents", "size"),
        avg_per_event=("costs.totalUsdCents", "mean"),
    ).reset_index().sort_values("total_usd_cents", ascending=False)
    display(by_model)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(by_model.head(15), x="total_usd_cents", y="modelId", hue="providerId", ax=ax)
    ax.set_title("Top 15 models by total cost (USD cents)")
    plt.tight_layout()
    plt.show()


## Top spenders

In [ ]:
if not df.empty:
    by_user = df.groupby("userId").agg(
        total_usd_cents=("costs.totalUsdCents", "sum"),
        events=("costs.totalUsdCents", "size"),
        prompt_tokens=("usage.promptTokens", "sum"),
        completion_tokens=("usage.completionTokens", "sum"),
    ).reset_index().sort_values("total_usd_cents", ascending=False)
    display(by_user.head(20))


## Budget vs actual

Joins `reactor_user_budgets` with the per-user totals to produce a budget-vs-actual table for the *current* day/week/month.

In [ ]:
if not df.empty:
    budget_rows = list(budgets.find({}, {"_id": 0, "userId": 1, "active": 1, "timezone": 1, "periods": 1}))
    if budget_rows:
        bdf = pd.json_normalize(budget_rows)
        # Compare totals against limit per period (UTC bucketing for simplicity here)
        rows = []
        for period_col, key in [("day", "periods.day.limitUsdCents"),
                                 ("week", "periods.week.limitUsdCents"),
                                 ("month", "periods.month.limitUsdCents")]:
            most_recent = df[period_col].max()
            spend = df[df[period_col] == most_recent].groupby("userId")["costs.totalUsdCents"].sum()
            for _, b in bdf.iterrows():
                limit = b.get(key)
                if not limit or pd.isna(limit):
                    continue
                used = spend.get(b["userId"], 0)
                rows.append({
                    "userId": b["userId"],
                    "period": period_col,
                    "limit_usd_cents": limit,
                    "used_usd_cents": used,
                    "pct_used": round((used / limit) * 100, 1) if limit else 0,
                })
        if rows:
            display(pd.DataFrame(rows).sort_values(["period", "pct_used"], ascending=[True, False]))
        else:
            print("No budgets configured.")
    else:
        print("No budgets configured.")


In [ ]:
# ── Optional: dump CSVs to OUTPUT_DIR ────────────────────────────────────────
if OUTPUT_DIR and not df.empty:
    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    by_conv.to_csv(out / "cost_per_conversation.csv", index=False)
    by_user.to_csv(out / "cost_per_user.csv", index=False)
    by_model.to_csv(out / "cost_per_model.csv", index=False)
    print(f"Wrote CSVs to {out}")


In [ ]:
client.close()
print("MongoDB connection closed.")